# Single MLP and its exact prediction equation
Train one 5-16-16-1 tanh MLP on the fixed 36 training cases, select alpha by
three-case-fold CV, diagnose withheld temperature/rate groups with nested alpha
selection, and finally score 37 test cases. Seed 42. All preprocessing is fitted
within fitting cases. The final model/equation is locked before test measurements
are opened by the modeling code. Historical test exposure remains disclosed.

Upload the professor ZIP to Drive and set ZIP_SOURCE below. Run cells in order;
restart the runtime if requested after installation. Outputs go to a new Drive
folder. The learned equation is empirical, with fixed assumed phi=0.04 metadata.
All numerical coefficients are saved in equation.json; inference requires only NumPy.
Source stress units remain unconfirmed. Hosted Colab execution is not verified.


In [ ]:
%pip install -q numpy==2.2.6 scipy==1.15.3 scikit-learn==1.7.2 joblib==1.6.0 matplotlib==3.10.9


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
ZIP_SOURCE=Path('/content/drive/MyDrive/MMM-ML/TEAM_8_extracted_results_set_1.zip')
LOCAL_ROOT=Path('/content/single_mlp_study')
OUTPUT=Path('/content/drive/MyDrive/MMM-ML/single_mlp_runs')/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')


In [ ]:
import json,sys,zipfile
for relative,content in {'src/single_mlp.py': '"""Fresh single-MLP experiment and exact empirical equation export; no physics loss."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport time\nimport warnings\n\nimport joblib\nimport numpy as np\nimport scipy\nimport sklearn\nfrom sklearn.model_selection import KFold\nfrom sklearn.neural_network import MLPRegressor\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef json_text(value):\n    def scalar(v):\n        if isinstance(v,np.generic): return v.item()\n        raise TypeError(f\'Unsupported JSON value: {type(v).__name__}\')\n    return json.dumps(value,indent=2,default=scalar,allow_nan=False)+\'\\n\'\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef load_cases(raw, ids, grid):\n    raw = Path(raw)\n    with (raw/\'simulation_matrix.csv\').open() as f:\n        matrix = {int(r[\'Simulation_ID\']):r for r in csv.DictReader(f)}\n    X, Y, audit = [], [], []\n    for i in ids:\n        path = raw/f\'simulation_{i:04d}\'/\'STR_STR.OUT\'\n        with path.open() as f:\n            assert f.readline().split() == \'TIME Evm Svm E11 E22 E33 E23 E13 E12 SCAU11 SCAU22 SCAU33 SCAU23 SCAU13 SCAU12 TEMP\'.split()\n        a = np.loadtxt(path, skiprows=1, usecols=(2,3))\n        if not np.isfinite(a).all() or not np.all(np.diff(a[:,1]) > 0):\n            raise ValueError(f\'Invalid curve {i}\')\n        if grid[0] < a[0,1] or grid[-1] > a[-1,1]:\n            raise ValueError(f\'Curve {i} would require extrapolation\')\n        r = matrix[i]\n        X.append([float(r[\'temperature\']),np.log10(float(r[\'strain_rate\'])),float(r[\'density\']),float(r[\'shear_modulus\'])])\n        Y.append(np.interp(grid,a[:,1],a[:,0]))\n        audit.append(dict(id=int(i),sha256=sha(path),rows=len(a)))\n    return np.array(X),np.array(Y),audit\n\n\ndef queries(X, grid):\n    return np.c_[np.repeat(X,len(grid),axis=0),np.tile(grid,len(X))]\n\n\ndef fit(X,Y,grid,alpha,cfg):\n    z = queries(X,grid)\n    xs = StandardScaler().fit(z)\n    ym,ys = float(Y.mean()),float(Y.std())\n    if ys <= 0:\n        raise ValueError(\'Response scale must be positive\')\n    net = MLPRegressor(hidden_layer_sizes=tuple(cfg[\'hidden\']),activation=\'tanh\',solver=\'lbfgs\',\n        alpha=alpha,random_state=cfg[\'seed\'],max_iter=cfg[\'max_iter\'],max_fun=cfg[\'max_fun\'],tol=cfg[\'tol\'])\n    with warnings.catch_warnings(record=True) as caught:\n        warnings.simplefilter(\'always\')\n        net.fit(xs.transform(z),(Y.ravel()-ym)/ys)\n    model = dict(net=net,xs=xs,ym=ym,ys=ys)\n    info = dict(alpha=alpha,iterations=int(net.n_iter_),loss=float(net.loss_),\n        warnings=[str(w.message) for w in caught],mean=xs.mean_.tolist(),scale=xs.scale_.tolist(),\n        target_mean=ym,target_scale=ys)\n    return model,info\n\n\ndef predict(model,X,grid):\n    return (model[\'net\'].predict(model[\'xs\'].transform(queries(X,grid)))*model[\'ys\']+model[\'ym\']).reshape(len(X),len(grid))\n\n\ndef export_equation(model):\n    n = model[\'net\']\n    return dict(input_order=[\'temperature\',\'log10_rate\',\'density\',\'G\',\'E11\'],\n        mean=model[\'xs\'].mean_.tolist(),scale=model[\'xs\'].scale_.tolist(),\n        target_mean=model[\'ym\'],target_scale=model[\'ys\'],\n        W1=n.coefs_[0].T.tolist(),b1=n.intercepts_[0].tolist(),\n        W2=n.coefs_[1].T.tolist(),b2=n.intercepts_[1].tolist(),\n        w3=n.coefs_[2][:,0].tolist(),b3=float(n.intercepts_[2][0]),\n        formula=\'S = target_mean + target_scale * (w3.T @ tanh(W2 @ tanh(W1 @ z + b1) + b2) + b3); z=(x-mean)/scale\',\n        interpretation=\'empirical prediction equation, not a physical governing law\')\n\n\ndef equation_predict(eq,z):\n    z = (np.asarray(z,dtype=float)-np.array(eq[\'mean\']))/np.array(eq[\'scale\'])\n    h1 = np.tanh(z@np.array(eq[\'W1\']).T+eq[\'b1\'])\n    h2 = np.tanh(h1@np.array(eq[\'W2\']).T+eq[\'b2\'])\n    return eq[\'target_mean\']+eq[\'target_scale\']*(h2@np.array(eq[\'w3\'])+eq[\'b3\'])\n\n\ndef lookup(X,Y,V):\n    out = []\n    for v in V:\n        same = (X[:,0]==v[0]) & (X[:,1]==v[1])\n        if not same.any(): same[:] = True\n        c = X[same]\n        d = np.abs(c[:,0]-v[0])/50+np.abs(c[:,1]-v[1])+np.abs(c[:,2]-v[2])/1000+np.abs(c[:,3]-v[3])/1000\n        out.append(Y[same][d.argmin()])\n    return np.array(out)\n\n\ndef scores(y,p):\n    rmse = np.sqrt(np.mean((y-p)**2,axis=1)); mae = np.mean(np.abs(y-p),axis=1)\n    norm = rmse/np.sqrt(np.mean(y*y,axis=1))\n    return dict(mean_rmse=float(rmse.mean()),median_rmse=float(np.median(rmse)),worst_rmse=float(rmse.max()),\n        mean_mae=float(mae.mean()),mean_nrmse=float(norm.mean()),worst_nrmse=float(norm.max()),\n        case_rmse=rmse.tolist(),case_mae=mae.tolist())\n\n\ndef select(X,Y,ids,grid,cfg,cache=None):\n    folds = list(KFold(cfg[\'folds\'],shuffle=True,random_state=cfg[\'seed\']).split(X))\n    trials = []; base = np.empty_like(Y)\n    for alpha in cfg[\'alphas\']:\n        cached=None if cache is None else cache/f\'alpha_{alpha}.json\'\n        if cached is not None and cached.exists():\n            trial=json.loads(cached.read_text());trial[\'prediction\']=np.array(trial[\'prediction\'])\n            trials.append(trial)\n            for tr,va in folds: base[va]=lookup(X[tr],Y[tr],X[va])\n            continue\n        p = np.empty_like(Y); records = []\n        for tr,va in folds:\n            m,info = fit(X[tr],Y[tr],grid,alpha,cfg)\n            p[va] = predict(m,X[va],grid); base[va] = lookup(X[tr],Y[tr],X[va])\n            records.append(dict(fit_ids=ids[tr].tolist(),validation_ids=ids[va].tolist(),**info))\n        trial=dict(alpha=alpha,metrics=scores(Y,p),folds=records,prediction=p)\n        trials.append(trial)\n        if cached is not None:\n            cached.parent.mkdir(parents=True,exist_ok=True)\n            cached.write_text(json_text({**trial,\'prediction\':p.tolist()}))\n        print(\'CV alpha finished\',alpha,\'cases\',len(ids),flush=True)\n    best = min(trials,key=lambda t:(t[\'metrics\'][\'mean_rmse\'],-t[\'alpha\']))\n    return best,trials,base\n\n\ndef run(root,output):\n    root,output = Path(root),Path(output)\n    output.mkdir(parents=True,exist_ok=False)\n    cfg = json.loads((root/\'configs/single_mlp.json\').read_text())\n    split = json.loads((root/\'data/split.json\').read_text())\n    train,test = np.array(split[\'train\']),np.array(split[\'test\'])\n    assert len(set(train))==36 and len(set(test))==37 and not set(train)&set(test)\n    grid = np.linspace(cfg[\'strain_min\'],cfg[\'strain_max\'],cfg[\'strain_points\'])\n    protected = {str(p.relative_to(root)):sha(p) for p in (root/\'runs/score/final_Svm\').glob(\'*\') if p.is_file()}\n    start = time.perf_counter()\n    X,Y,audit = load_cases(root/\'TEAM_8_extracted_results_set_1\',train,grid)\n    signature=hashlib.sha256((sha(__file__)+json_text(cfg)+json_text(audit)).encode()).hexdigest()[:16]\n    cache=root/\'work\'/f\'single_mlp_cache_{signature}\'\n    best,trials,base = select(X,Y,train,grid,cfg,cache/\'primary\')\n    print(\'Training-only selection:\',best[\'alpha\'],best[\'metrics\'][\'mean_rmse\'],flush=True)\n    groups = []\n    for T,rate in sorted(set(map(tuple,X[:,:2]))):\n        group_cache=cache/f\'group_{T}_{rate}\'\n        group_result=group_cache/\'result.json\'\n        if group_result.exists():\n            groups.append(json.loads(group_result.read_text()))\n            print(\'Restored completed condition check\',T,rate,flush=True)\n            continue\n        va = np.flatnonzero((X[:,0]==T)&(X[:,1]==rate)); tr = np.setdiff1d(np.arange(36),va)\n        b,ts,_ = select(X[tr],Y[tr],train[tr],grid,cfg,group_cache)\n        m,info = fit(X[tr],Y[tr],grid,b[\'alpha\'],cfg)\n        groups.append(dict(temperature=T,log10_rate=rate,fit_ids=train[tr].tolist(),validation_ids=train[va].tolist(),\n            alpha=b[\'alpha\'],selection=[{k:v for k,v in t.items() if k!=\'prediction\'} for t in ts],\n            fit=info,mlp=scores(Y[va],predict(m,X[va],grid)),baseline=scores(Y[va],lookup(X[tr],Y[tr],X[va]))))\n        group_result.write_text(json_text(groups[-1]))\n        print(\'Nested condition check\',T,rate,groups[-1][\'mlp\'][\'mean_rmse\'],flush=True)\n    model,fit_info = fit(X,Y,grid,best[\'alpha\'],cfg)\n    eq = export_equation(model)\n    (output/\'equation.json\').write_text(json_text(eq))\n    joblib.dump(model,output/\'model.joblib\')\n    lock = dict(config=cfg,alpha=best[\'alpha\'],fit_ids=train.tolist(),seed=cfg[\'seed\'],\n                equation_sha256=sha(output/\'equation.json\'),model_sha256=sha(output/\'model.joblib\'))\n    (output/\'selection_lock.json\').write_text(json_text(lock))\n    # No test history is opened until the selected fitted model has been saved and locked.\n    V,Z,test_audit = load_cases(root/\'TEAM_8_extracted_results_set_1\',test,grid)\n    p,pb = predict(model,V,grid),lookup(X,Y,V)\n    portable = equation_predict(json.loads((output/\'equation.json\').read_text()),queries(V,grid)).reshape(Z.shape)\n    reloaded = predict(joblib.load(output/\'model.joblib\'),V,grid)\n    err = float(np.max(np.abs(portable-p))); reload_err = float(np.max(np.abs(reloaded-p)))\n    assert err <= 1e-10 and reload_err == 0.\n    for rel,h in protected.items(): assert sha(root/rel)==h\n    result = dict(config=cfg,train_ids=train.tolist(),test_ids=test.tolist(),training_audit=audit,test_audit=test_audit,\n        trials=[{k:v for k,v in t.items() if k!=\'prediction\'} for t in trials],selected_alpha=best[\'alpha\'],\n        cv=best[\'metrics\'],cv_baseline=scores(Y,base),condition_holdouts=groups,final_fit=fit_info,\n        test=scores(Z,p),test_baseline=scores(Z,pb),train=scores(Y,predict(model,X,grid)),\n        test_by_condition=[dict(temperature=T,log10_rate=rate,metrics=scores(Z[(V[:,0]==T)&(V[:,1]==rate)],p[(V[:,0]==T)&(V[:,1]==rate)])) for T,rate in sorted(set(map(tuple,V[:,:2])))],\n        checks=dict(equation_max_error=err,reload_max_error=reload_err,parameters=sum(w.size for w in model[\'net\'].coefs_)+sum(b.size for b in model[\'net\'].intercepts_),\n            protected_artifacts_unchanged=protected,selection_saved_before_test=True),\n        versions=dict(python=platform.python_version(),numpy=np.__version__,scipy=scipy.__version__,sklearn=sklearn.__version__),\n        elapsed_seconds=time.perf_counter()-start,code_sha256=sha(__file__),split_sha256=sha(root/\'data/split.json\'),\n        caveat=\'Previously explored test set; same-design interpolation, not blind/physical validation. Condition checks use nested training-only alpha selection. No universal-best claim.\')\n    (output/\'results.json\').write_text(json_text(result))\n    for label,ids,y,pr,bl in [(\'cv\',train,Y,best[\'prediction\'],base),(\'test\',test,Z,p,pb)]:\n        with (output/f\'{label}_predictions.csv\').open(\'w\',newline=\'\') as f:\n            w=csv.writer(f);w.writerow([\'id\',\'E11\',\'Svm\',\'MLP\',\'lookup\'])\n            for i,case in enumerate(ids):\n                w.writerows(zip([case]*len(grid),grid,y[i],pr[i],bl[i]))\n    cache=root/\'work/matplotlib\';cache.mkdir(parents=True,exist_ok=True);os.environ.setdefault(\'MPLCONFIGDIR\',str(cache))\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig,axes=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)\n    worst=int(np.argmax(result[\'test\'][\'case_rmse\']));i=list(train).index(60)\n    for ax,e,truth,mp,bp,title in [(axes[0],grid,Z[worst],p[worst],pb[worst],f\'Worst test case {test[worst]}\'),(axes[1],grid,Y[i],best[\'prediction\'][i],base[i],\'Case 60: training-side case validation\')]:\n        ax.plot(e,truth,label=\'Supplied curve\',color=\'black\');ax.plot(e,mp,label=\'Single MLP\');ax.plot(e,bp,\'--\',label=\'Curve-copy baseline\')\n        ax.set(title=title,xlabel=\'E11 (source strain coordinate)\',ylabel=\'Svm (source stress units)\');ax.legend()\n    fig.savefig(output/\'comparison.png\',dpi=150);plt.close(fig)\n    print(json_text({k:result[k] for k in [\'selected_alpha\',\'cv\',\'test\',\'test_baseline\',\'checks\']}),flush=True)\n    return result\n\n\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser();ap.add_argument(\'--root\',type=Path,default=ROOT);ap.add_argument(\'--output\',type=Path,required=True)\n    args=ap.parse_args()\n    with threadpool_limits(limits=1): run(args.root,args.output)\n', 'configs/single_mlp.json': '{\n  "hidden": [16, 16], "activation": "tanh", "solver": "lbfgs",\n  "seed": 42, "folds": 3, "alphas": [0.0001, 0.001, 0.01],\n  "max_iter": 5000, "max_fun": 50000, "tol": 1e-9,\n  "strain_min": 0.0002, "strain_max": 0.0999, "strain_points": 120,\n  "porosity_fraction_assumed": 0.04\n}\n', 'data/split.json': '{\n "seed": 42,\n "train": [\n  2,\n  3,\n  5,\n  6,\n  8,\n  18,\n  19,\n  21,\n  22,\n  25,\n  26,\n  27,\n  28,\n  29,\n  30,\n  33,\n  34,\n  38,\n  40,\n  41,\n  43,\n  46,\n  49,\n  52,\n  53,\n  60,\n  64,\n  65,\n  66,\n  69,\n  70,\n  72,\n  76,\n  78,\n  80,\n  81\n ],\n "test": [\n  1,\n  4,\n  7,\n  9,\n  10,\n  11,\n  12,\n  13,\n  14,\n  15,\n  16,\n  17,\n  20,\n  23,\n  24,\n  31,\n  32,\n  35,\n  36,\n  37,\n  39,\n  42,\n  44,\n  45,\n  47,\n  48,\n  50,\n  51,\n  54,\n  67,\n  68,\n  71,\n  73,\n  74,\n  75,\n  77,\n  79\n ]\n}'}.items():
    p=LOCAL_ROOT/relative
    p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(content,encoding='utf-8')
split=json.loads((LOCAL_ROOT/'data/split.json').read_text())
raw=LOCAL_ROOT/'TEAM_8_extracted_results_set_1'
with zipfile.ZipFile(ZIP_SOURCE) as archive:
    for relative in ['simulation_matrix.csv']+[f'simulation_{i:04d}/STR_STR.OUT' for i in split['train']+split['test']]:
        matches=[p for p in archive.namelist() if p.replace('\\','/')==relative or p.replace('\\','/').endswith('/'+relative)]
        if len(matches)!=1:raise ValueError(relative)
        p=raw/relative;p.parent.mkdir(parents=True,exist_ok=True)
        p.write_bytes(archive.read(matches[0]))
sys.path.insert(0,str(LOCAL_ROOT/'src'))
import single_mlp
from threadpoolctl import threadpool_limits
with threadpool_limits(limits=1):
    result=single_mlp.run(LOCAL_ROOT,OUTPUT)
print('Saved:',OUTPUT)


In [ ]:
from IPython.display import display,Image
print('Training-only CV RMSE:',result['cv']['mean_rmse'])
print('Test RMSE:',result['test']['mean_rmse'])
print('Baseline test RMSE:',result['test_baseline']['mean_rmse'])
print('Equation equivalence:',result['checks']['equation_max_error'])
display(Image(filename=str(OUTPUT/'comparison.png')))


## Use the numerical equation
For raw inputs x=[T, log10(rate), density, G, E11], compute z=(x-mean)/scale,
h1=tanh(W1 z+b1), h2=tanh(W2 h1+b2), and
Svm=target_mean+target_scale*(w3.T h2+b3).
The JSON contains every coefficient. `single_mlp.equation_predict(eq, queries)`
evaluates batches of five-column inputs. This is the exact trained network formula.
No physical governing law, experimental validation or porosity dependence is implied.
Avoid claims beyond the observed loading family and E11=0.0002 to 0.0999.
